# Demo 2D Image -> 3D tren Google Colab GPU T4

Notebook nay upload 1 anh JPG/PNG, uoc luong depth bang Depth Anything V2, tao point cloud va mesh bang Open3D, sau do xuat `.ply`, `.obj`, `.glb` de preview bang `model-viewer`.

Luu y: pipeline depth -> mesh chi tao duoc hinh hoc tu mot goc nhin, nen rat hop cho demo nhanh, phong canh, mat truoc vat the. Neu can object 360 hoan chinh, hay thay block mesh bang TripoSR, Zero123++, Shap-E hoac model image-to-3D chuyen dung.

In [ ]:
# Cai dat thu vien cho Colab free GPU. Neu Colab da co torch phu hop, dong cai torch co the bo qua.
!pip -q install --upgrade pip
!pip -q install torch torchvision --index-url https://download.pytorch.org/whl/cu124
!pip -q install transformers accelerate timm opencv-python pillow matplotlib numpy open3d trimesh fastapi uvicorn pyngrok python-multipart nest_asyncio

In [ ]:
# Import va cau hinh chung
import json
import os
import re
import uuid
from base64 import b64encode
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import open3d as o3d
import torch
import trimesh
from google.colab import files
from IPython.display import HTML, display
from PIL import Image
from transformers import AutoImageProcessor, AutoModelForDepthEstimation

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MAX_IMAGE_SIDE = 768       # Giam xuong 512 neu bi het RAM/VRAM.
POISSON_DEPTH = 8          # Giam xuong 7 de nhanh hon, tang len 9 neu may du manh.
TARGET_TRIANGLES = 60000   # Gioi han so tam giac khi export GLB/OBJ.
VOXEL_SIZE = 0.005         # Tang len 0.008-0.012 neu point cloud qua nang.

BASE_DIR = Path('/content/image2_3d_demo')
INPUT_DIR = BASE_DIR / 'inputs'
OUTPUT_DIR = BASE_DIR / 'outputs'
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Device: {DEVICE}')
print(f'Output dir: {OUTPUT_DIR}')

In [ ]:
# Load model pretrained nhe cho Colab T4.
# Co the doi sang: 'depth-anything/Depth-Anything-V2-Base-hf' de depth tot hon nhung cham/nang hon.
MODEL_ID = 'depth-anything/Depth-Anything-V2-Small-hf'
processor = AutoImageProcessor.from_pretrained(MODEL_ID)
depth_model = AutoModelForDepthEstimation.from_pretrained(MODEL_ID).to(DEVICE)
if DEVICE == 'cuda':
    depth_model = depth_model.half()
depth_model.eval()
print('Da load model:', MODEL_ID)

In [ ]:
# Cac ham xu ly chinh: anh -> depth -> point cloud -> mesh -> file 3D
def safe_stem(path_or_name: str) -> str:
    stem = Path(path_or_name).stem
    stem = re.sub(r'[^A-Za-z0-9_.-]+', '_', stem).strip('._')
    return stem or 'image'


def open_and_resize_image(image_path: str, max_side: int = MAX_IMAGE_SIDE) -> Image.Image:
    """Mo anh RGB va resize de hop voi RAM/VRAM gioi han cua Colab free."""
    image_pil = Image.open(image_path).convert('RGB')
    w, h = image_pil.size
    scale = min(max_side / max(w, h), 1.0)
    if scale < 1.0:
        image_pil = image_pil.resize((int(w * scale), int(h * scale)), Image.Resampling.LANCZOS)
    return image_pil


def estimate_depth(image_pil: Image.Image) -> np.ndarray:
    """Tra ve depth map chuan hoa trong khoang [0, 1]."""
    inputs = processor(images=image_pil, return_tensors='pt')
    inputs = {key: value.to(DEVICE) for key, value in inputs.items()}

    with torch.no_grad():
        if DEVICE == 'cuda':
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                outputs = depth_model(**inputs)
        else:
            outputs = depth_model(**inputs)

    pred_depth = outputs.predicted_depth
    pred_depth = torch.nn.functional.interpolate(
        pred_depth.unsqueeze(1),
        size=image_pil.size[::-1],
        mode='bicubic',
        align_corners=False,
    ).squeeze()

    depth = pred_depth.detach().cpu().numpy()
    depth = (depth - depth.min()) / (depth.max() - depth.min() + 1e-8)
    return depth.astype(np.float32)


def depth_to_point_cloud(image_rgb: np.ndarray, depth_norm: np.ndarray) -> o3d.geometry.PointCloud:
    """Tao point cloud mau tu RGB + depth bang camera pinhole gia lap."""
    h, w = depth_norm.shape
    depth_for_3d = 1.0 - depth_norm
    depth_u16 = np.clip(depth_for_3d * 1200.0, 1, 65535).astype(np.uint16)

    color_o3d = o3d.geometry.Image(image_rgb)
    depth_o3d = o3d.geometry.Image(depth_u16)
    fx = fy = max(w, h) * 1.2
    intrinsic = o3d.camera.PinholeCameraIntrinsic(w, h, fx, fy, w / 2.0, h / 2.0)

    rgbd = o3d.geometry.RGBDImage.create_from_color_and_depth(
        color_o3d,
        depth_o3d,
        depth_scale=1000.0,
        depth_trunc=5.0,
        convert_rgb_to_intensity=False,
    )
    pcd = o3d.geometry.PointCloud.create_from_rgbd_image(rgbd, intrinsic)
    pcd.transform([[1, 0, 0, 0], [0, -1, 0, 0], [0, 0, -1, 0], [0, 0, 0, 1]])
    pcd = pcd.voxel_down_sample(voxel_size=VOXEL_SIZE)
    pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=1.5)
    return pcd


def colorize_mesh_from_point_cloud(mesh: o3d.geometry.TriangleMesh, pcd: o3d.geometry.PointCloud) -> o3d.geometry.TriangleMesh:
    """Gan mau vertex cua mesh theo diem point cloud gan nhat."""
    if not pcd.has_colors() or len(mesh.vertices) == 0:
        return mesh
    kdtree = o3d.geometry.KDTreeFlann(pcd)
    pcd_colors = np.asarray(pcd.colors)
    vertex_colors = []
    for vertex in np.asarray(mesh.vertices):
        _, idx, _ = kdtree.search_knn_vector_3d(vertex, 1)
        vertex_colors.append(pcd_colors[idx[0]] if idx else [0.8, 0.8, 0.8])
    mesh.vertex_colors = o3d.utility.Vector3dVector(np.asarray(vertex_colors))
    return mesh


def point_cloud_to_mesh(pcd: o3d.geometry.PointCloud) -> o3d.geometry.TriangleMesh:
    """Tai tao mesh bang Poisson reconstruction va don dep mesh truoc khi xuat."""
    if len(pcd.points) < 100:
        raise ValueError('Point cloud qua it diem. Hay thu anh ro hon hoac giam VOXEL_SIZE.')

    pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.03, max_nn=30))
    pcd.orient_normals_consistent_tangent_plane(10)
    mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=POISSON_DEPTH)

    densities = np.asarray(densities)
    mesh.remove_vertices_by_mask(densities < np.quantile(densities, 0.05))
    mesh = mesh.crop(pcd.get_axis_aligned_bounding_box())

    if len(mesh.triangles) > TARGET_TRIANGLES:
        mesh = mesh.simplify_quadric_decimation(target_number_of_triangles=TARGET_TRIANGLES)

    mesh.remove_degenerate_triangles()
    mesh.remove_duplicated_triangles()
    mesh.remove_duplicated_vertices()
    mesh.remove_non_manifold_edges()
    mesh.compute_vertex_normals()
    return colorize_mesh_from_point_cloud(mesh, pcd)


def save_outputs(image_name: str, depth_norm: np.ndarray, pcd: o3d.geometry.PointCloud, mesh: o3d.geometry.TriangleMesh) -> dict:
    out_dir = OUTPUT_DIR / f'{safe_stem(image_name)}_{uuid.uuid4().hex[:8]}'
    out_dir.mkdir(parents=True, exist_ok=True)

    depth_png = out_dir / 'depth.png'
    ply_path = out_dir / 'point_cloud.ply'
    obj_path = out_dir / 'mesh.obj'
    glb_path = out_dir / 'mesh.glb'

    cv2.imwrite(str(depth_png), (depth_norm * 255).astype(np.uint8))
    o3d.io.write_point_cloud(str(ply_path), pcd)
    o3d.io.write_triangle_mesh(str(obj_path), mesh, write_triangle_uvs=False)

    colors = None
    if mesh.has_vertex_colors():
        colors = (np.asarray(mesh.vertex_colors) * 255).clip(0, 255).astype(np.uint8)
    tm = trimesh.Trimesh(
        vertices=np.asarray(mesh.vertices),
        faces=np.asarray(mesh.triangles),
        vertex_normals=np.asarray(mesh.vertex_normals) if len(mesh.vertex_normals) else None,
        vertex_colors=colors,
        process=False,
    )
    tm.export(str(glb_path))

    return {'out_dir': str(out_dir), 'depth_png': str(depth_png), 'ply': str(ply_path), 'obj': str(obj_path), 'glb': str(glb_path)}


def run_pipeline(image_path: str):
    image_pil = open_and_resize_image(image_path)
    image_rgb = np.array(image_pil)
    depth_norm = estimate_depth(image_pil)
    pcd = depth_to_point_cloud(image_rgb, depth_norm)
    mesh = point_cloud_to_mesh(pcd)
    artifacts = save_outputs(Path(image_path).name, depth_norm, pcd, mesh)
    return image_rgb, depth_norm, pcd, mesh, artifacts

In [ ]:
# Upload anh va chay pipeline tu dau den cuoi
uploaded = files.upload()
assert len(uploaded) > 0, 'Ban can upload it nhat 1 file anh JPG/PNG.'

first_name = next(iter(uploaded.keys()))
input_path = INPUT_DIR / first_name
with open(input_path, 'wb') as f:
    f.write(uploaded[first_name])

image_rgb, depth_norm, pcd, mesh, artifacts = run_pipeline(str(input_path))
print(json.dumps(artifacts, indent=2))
print(f'Point cloud: {len(pcd.points)} points')
print(f'Mesh: {len(mesh.vertices)} vertices, {len(mesh.triangles)} triangles')

In [ ]:
# Hien thi anh goc va depth map
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.title('Anh goc')
plt.imshow(image_rgb)
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title('Depth map')
plt.imshow(depth_norm, cmap='inferno')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Preview GLB ngay trong Colab bang model-viewer
with open(artifacts['glb'], 'rb') as f:
    glb_b64 = b64encode(f.read()).decode('utf-8')

display(HTML(f'''
<script type="module" src="https://unpkg.com/@google/model-viewer/dist/model-viewer.min.js"></script>
<model-viewer src="data:model/gltf-binary;base64,{glb_b64}" camera-controls auto-rotate shadow-intensity="1" style="width:100%;height:520px;background:#eef2f7;border:1px solid #d8dee8;border-radius:8px;"></model-viewer>
'''))

In [ ]:
# Tai cac file ket qua ve may
files.download(artifacts['glb'])
files.download(artifacts['obj'])
files.download(artifacts['ply'])
files.download(artifacts['depth_png'])

## Tuy chon: mo API va website public ngay trong Colab

Cell duoi tao FastAPI server nho, expose form upload va file output qua ngrok. Neu tai khoan ngrok cua ban bat buoc token, dat `NGROK_AUTHTOKEN` truoc khi chay cell.

In [ ]:
# FastAPI + HTML frontend mini trong Colab
import nest_asyncio
import uvicorn
from fastapi import FastAPI, File, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import HTMLResponse
from fastapi.staticfiles import StaticFiles
from pyngrok import ngrok

nest_asyncio.apply()

token = os.getenv('NGROK_AUTHTOKEN')
if token:
    ngrok.set_auth_token(token)

app = FastAPI(title='Image2To3D Colab Demo')
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_credentials=True, allow_methods=['*'], allow_headers=['*'])
app.mount('/outputs', StaticFiles(directory=str(OUTPUT_DIR)), name='outputs')

HTML_PAGE = '''
<!doctype html><html lang="vi"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>2D sang 3D Colab</title><script type="module" src="https://unpkg.com/@google/model-viewer/dist/model-viewer.min.js"></script>
<style>body{margin:0;font-family:Arial,sans-serif;background:#f7f7f4;color:#1c2430}.wrap{max-width:980px;margin:32px auto;padding:24px;background:white;border:1px solid #d8dee8;border-radius:8px}form{display:grid;grid-template-columns:1fr auto;gap:10px;margin:16px 0}button{border:0;background:#2563eb;color:white;border-radius:8px;padding:12px 16px;font-weight:700}input{border:1px dashed #d8dee8;border-radius:8px;padding:10px}.status{padding:10px 12px;background:#eef2f7;border-radius:8px}.grid{display:grid;grid-template-columns:1fr 1.3fr;gap:14px;margin-top:14px}img,model-viewer{width:100%;height:420px;object-fit:contain;background:#eef2f7;border:1px solid #d8dee8;border-radius:8px}.links{display:flex;gap:10px;flex-wrap:wrap;margin-top:14px}.links a{background:#0f766e;color:white;text-decoration:none;border-radius:8px;padding:8px 12px}@media(max-width:720px){form,.grid{grid-template-columns:1fr}img,model-viewer{height:320px}}</style></head>
<body><main class="wrap"><h1>2D sang 3D</h1><p>Upload JPG/PNG de tao depth map, point cloud va mesh GLB/OBJ/PLY.</p><form id="form"><input id="file" name="file" type="file" accept="image/png,image/jpeg" required><button id="btn">Tao mo hinh 3D</button></form><div id="status" class="status">San sang.</div><div class="grid"><img id="preview" alt="Anh dau vao"><model-viewer id="viewer" camera-controls auto-rotate shadow-intensity="1"></model-viewer></div><div id="links" class="links"></div></main>
<script>const f=document.getElementById('file'),form=document.getElementById('form'),s=document.getElementById('status'),v=document.getElementById('viewer'),p=document.getElementById('preview'),links=document.getElementById('links'),btn=document.getElementById('btn');f.onchange=()=>{if(f.files[0])p.src=URL.createObjectURL(f.files[0])};form.onsubmit=async e=>{e.preventDefault();btn.disabled=true;s.textContent='Dang xu ly 3D...';links.innerHTML='';try{const fd=new FormData();fd.append('file',f.files[0]);const r=await fetch('/api/generate',{method:'POST',body:fd});const d=await r.json();if(!r.ok)throw Error(d.detail||'Loi xu ly');v.src=d.preview_glb_url+'?t='+Date.now();s.textContent='Da tao xong mo hinh 3D.';for(const [k,u] of Object.entries(d.download)){const a=document.createElement('a');a.href=u;a.target='_blank';a.textContent='Tai '+k.toUpperCase();links.appendChild(a)}}catch(err){s.textContent='Loi: '+err.message}finally{btn.disabled=false}}</script></body></html>
'''

@app.get('/', response_class=HTMLResponse)
async def home():
    return HTML_PAGE

@app.post('/api/generate')
async def api_generate(file: UploadFile = File(...)):
    ext = Path(file.filename or '').suffix.lower()
    if ext not in ['.jpg', '.jpeg', '.png']:
        raise HTTPException(status_code=400, detail='Chi ho tro JPG/PNG')
    input_path = INPUT_DIR / f'api_{uuid.uuid4().hex[:8]}{ext}'
    with open(input_path, 'wb') as f:
        f.write(await file.read())
    _, _, _, _, out = run_pipeline(str(input_path))
    job_id = Path(out['out_dir']).name
    return {
        'message': 'OK',
        'preview_glb_url': f'/outputs/{job_id}/mesh.glb',
        'download': {
            'glb': f'/outputs/{job_id}/mesh.glb',
            'obj': f'/outputs/{job_id}/mesh.obj',
            'ply': f'/outputs/{job_id}/point_cloud.ply',
            'depth_png': f'/outputs/{job_id}/depth.png',
        },
    }

public_url = ngrok.connect(8000).public_url
print('Public URL:', public_url)
uvicorn.run(app, host='0.0.0.0', port=8000)